In [0]:
from pyspark.sql.functions import *

spark.sql("USE CATALOG Silver_Catalog")
spark.sql("USE SCHEMA Silver_SCH")

tariff_df = spark.table("Bronze_Catalog.Bronze_SCH.Bronze_Tariff_Metrics")

In [0]:
# Step 2

tariff_df.printSchema()

In [0]:
# Step 4
print(tariff_df.count())

In [0]:
# Step 5

from pyspark.sql.functions import col, sum

null_df = tariff_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in tariff_df.columns
])

display(null_df)

In [0]:
from pyspark.sql.functions import expr

tariff_df = tariff_df \
.withColumn("unit_rate", expr("try_cast(unit_rate as double)")) \
.withColumn("peak_rate", expr("try_cast(peak_rate as double)")) \
.withColumn("offpeak_rate", expr("try_cast(offpeak_rate as double)")) \
.withColumn("fixed_charge", expr("try_cast(fixed_charge as double)")) \
.withColumn("tax_amount", expr("try_cast(tax_amount as double)")) \
.withColumn("subsidy_amount", expr("try_cast(subsidy_amount as double)")) \
.withColumn("monthly_bill", expr("try_cast(monthly_bill as double)")) \
.withColumn("billing_units", expr("try_cast(billing_units as double)")) \
.withColumn("late_fee", expr("try_cast(late_fee as double)")) \
.withColumn("adjustment_amount", expr("try_cast(adjustment_amount as double)"))

In [0]:
# Fill Nulls
tariff_df = tariff_df.fillna({
    "unit_rate": 0,
    "peak_rate": 0,
    "offpeak_rate": 0,
    "fixed_charge": 0,
    "tax_amount": 0,
    "subsidy_amount": 0,
    "monthly_bill": 0,
    "billing_units": 0,
    "late_fee": 0,
    "adjustment_amount": 0
})

In [0]:
tariff_df.printSchema()

In [0]:
# Data Validation

from pyspark.sql.functions import col

tariff_df = tariff_df.filter(
    (col("unit_rate") >= 0) &
    (col("peak_rate") >= 0) &
    (col("offpeak_rate") >= 0) &
    (col("fixed_charge") >= 0) &
    (col("tax_amount") >= 0) &
    (col("subsidy_amount") >= 0) &
    (col("monthly_bill") >= 0) &
    (col("billing_units") >= 0) &
    (col("late_fee") >= 0)
)

print("Valid Records :", tariff_df.count())

In [0]:
# Step 10 - Create Surrogate Key
from pyspark.sql.functions import monotonically_increasing_id

tariff_df = tariff_df.withColumn(
    "tariff_sk",
    monotonically_increasing_id()
)



In [0]:
# Step 11 - Implement SCD Type 2
from pyspark.sql.functions import current_timestamp, lit

tariff_df = tariff_df \
.withColumn("effective_from", current_timestamp()) \
.withColumn("effective_to", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(True))



In [0]:
# Step 12 - Save Silver Table
tariff_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("Silver_Catalog.Silver_SCH.Silver_Tariff_Metrics")

In [0]:
from pyspark.sql.functions import current_timestamp

records_loaded = tariff_df.count()

try:

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Tariff_Metrics',
        'Tariff Silver Pipeline',
        current_timestamp(),
        {records_loaded},
        'SUCCESS',
        NULL
    )
    """)

    print("✅ Silver_Tariff_Metrics Loaded Successfully")

   

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Tariff_Metrics',
        'Tariff Silver Pipeline',
        current_timestamp(),
        0,
        'FAILED',
        '{error_message}'
    )
    """)

    print(f"❌ Silver_Tariff_Metrics Failed : {error_message}")

    

    raise

In [0]:
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Tariff_Metrics
""")

print("✅ OPTIMIZE Completed")

In [0]:
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Tariff_Metrics
ZORDER BY (household_id)
""")

print("✅ ZORDER Completed")

In [0]:
spark.sql("""
VACUUM Silver_Catalog.Silver_SCH.Silver_Tariff_Metrics
RETAIN 168 HOURS
""")

print("✅ VACUUM Completed")

In [0]:
spark.sql("""
DESCRIBE HISTORY Silver_Catalog.Silver_SCH.Silver_Tariff_Metrics
""").show(truncate=False)